# Run the search agent on Colab

Compute only: the same scripts as in Docker, with Ollama installed directly because Colab has no Docker.
Runtime: **Runtime > Change runtime type > T4 GPU**. Run the cells in order.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/stopping_policies_results'
REPO_URL = 'https://github.com/esarkarov/stopping_policies_project.git'
MODEL = 'qwen3.5:4b'
OLLAMA_VERSION = '0.40.5'
TAG = 'freeze-v1'

Mounted at /content/drive


In [ ]:
%cd /content
!rm -rf /content/project
!git clone {REPO_URL} /content/project
%cd /content/project
!git checkout {TAG}
!git describe --tags
!pip install -q -r requirements.txt gdown pyarrow

/content
Cloning into '/content/project'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 108 (delta 34), reused 94 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 547.56 KiB | 14.80 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/project
Note: switching to 'freeze-v1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at eeba30c chore: track pilot trajectories, tables and figure

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull {MODEL}
!ollama --version

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

ollama version is 0.40.5


In [ ]:
!mkdir -p {RESULTS}
!rm -rf results && ln -s {RESULTS} results
!python scripts/download_data.py
!python scripts/prepare_data.py

HotpotQA: downloading http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json
HotpotQA: official server unavailable (<urlopen error [Errno -2] Name or service not known>); using the Hugging Face copy
HotpotQA: converted 7405 questions to the official JSON format
HotpotQA: saved /content/project/data/raw/hotpot_dev_distractor_v1.json (46.4 MB)
MuSiQue: downloading from Google Drive
Downloading...
From (original): https://drive.google.com/uc?id=1tGdADlNjWFaHLeZZGShh2IRcpO6Lv24h
From (redirected): https://drive.google.com/uc?id=1tGdADlNjWFaHLeZZGShh2IRcpO6Lv24h&confirm=t&uuid=9717ad90-6319-496b-9efa-f1241dafe01d
To: /content/project/data/raw/musique_data_v1.0.zip
100% 272M/272M [00:01<00:00, 148MB/s]
MuSiQue: saved /content/project/data/raw/musique_ans_v1.0_dev.jsonl (30.4 MB)
{'dataset': 'hotpotqa', 'seed': 42, 'dev_questions': 7405, 'corpus_passages': 66635, 'pilot_questions': 30, 'main_questions': 300}
hotpotqa: BM25 index ready, query 'Lowe's' -> ['Eddie Lowe (footballe

In [ ]:
# expected: HotpotQA 7405 / 66635 passages, MuSiQue 2417 / 21100 passages
!python scripts/speed_test.py
!ollama ps

Model: qwen3.5:4b
think=False |  121.1 s | 'Answer: Christopher Nolan'
think=False |    0.2 s | 'Answer: Christopher Nolan'
think=True  |    4.3 s | 'Answer: Christopher Nolan'
NAME          ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen3.5:4b    d8b0f5e9760c    3.1 GB    100% GPU     4096       llamacpp    4 minutes from now    


## Logprobs check

Needed for the P1 P(yes) sweep. If the reply below has a `logprobs` field, the installed Ollama supports it.

In [ ]:
!ollama --version
!curl -s localhost:11434/api/chat -d '{"model":"qwen3.5:4b","messages":[{"role":"user","content":"Reply Yes"}],"stream":false,"think":false,"logprobs":true,"top_logprobs":5,"options":{"num_predict":2}}' | head -c 600

In [ ]:
# timing check on 2 questions (about 30 s each on a T4)
!python scripts/run_agent.py --dataset musique --split pilot --limit 2

## Pilot rerun (already done with freeze-v1; skip when only running the main set)

In [ ]:
!python scripts/run_agent.py --dataset hotpotqa --split pilot
!python scripts/run_agent.py --dataset musique --split pilot

In [ ]:
!python scripts/run_eval.py --dataset hotpotqa --split pilot
!python scripts/run_eval.py --dataset musique --split pilot

## Main runs (prompts and policies frozen as freeze-v1)

Safe to stop and rerun: finished questions are skipped, and results are kept in Google Drive.

In [ ]:
!python scripts/run_agent.py --dataset hotpotqa --split main

hotpotqa/main: 0 done, 300 to run, model qwen3.5:4b
100% 300/300 [1:37:37<00:00, 19.52s/it]
finished 300 questions in 5857s (19.5s per question)
saved to /content/project/results/trajectories/hotpotqa_main.jsonl


In [ ]:
!python scripts/run_agent.py --dataset musique --split main

musique/main: 0 done, 300 to run, model qwen3.5:4b
100% 300/300 [1:34:19<00:00, 18.87s/it]
finished 300 questions in 5660s (18.9s per question)
saved to /content/project/results/trajectories/musique_main.jsonl


## Confirmation runs (new questions, agent code unchanged)

Set `TAG = 'confirm-v1'` in the settings cell, then rerun the clone cell and the data cell (and the Ollama cell after a new runtime). Safe to stop and rerun: finished questions are skipped.

In [ ]:
!git diff --stat freeze-v1 HEAD -- src/agent.py src/prompts.py src/llm.py src/retriever.py
import json
ids = lambda path: [json.loads(line)['id'] for line in open(path)]
for ds in ('hotpotqa', 'musique'):
    main, pilot, confirm = (ids(f'data/processed/{ds}/{s}.jsonl') for s in ('main', 'pilot', 'confirm'))
    done = ids(f'results/trajectories/{ds}_main.jsonl')
    assert set(main) == set(done), ds
    assert len(confirm) == 400 and not set(confirm) & (set(main) | set(pilot)), ds
print('confirmation split ok (no file listed above means the agent code is identical to freeze-v1)')

In [ ]:
!python scripts/run_agent.py --dataset hotpotqa --split confirm

In [ ]:
!python scripts/run_agent.py --dataset musique --split confirm

## P1 check replay with P(yes)

Replays the P1 check on the saved passages and stores the probability of "Yes" for every step. Needs the trajectories in Drive and a tag that contains `scripts/run_p1_logprobs.py`. Safe to stop and rerun.

In [ ]:
!python scripts/run_p1_logprobs.py --dataset hotpotqa --split main

In [ ]:
!python scripts/run_p1_logprobs.py --dataset musique --split main

In [ ]:
!python scripts/run_p1_logprobs.py --dataset hotpotqa --split confirm

In [ ]:
!python scripts/run_p1_logprobs.py --dataset musique --split confirm